# 🚦 MTA Subway Real-Time Feed - Data Ingest
### Powered by Elasticsearch Serverless

This notebook pulls the MTA's **live subway feed** (GTFS-Realtime) and indexes upcoming train arrivals into Elasticsearch - a snapshot of where every train is *right now*.

**What this notebook does:**
1. Fetches the live protobuf feed from the [MTA real-time API](https://api.mta.info/#/subwayRealTimeFeeds) - **no API key required** (as of 2023)
2. Parses it with the official `gtfs-realtime-bindings`
3. Extracts upcoming arrival predictions per stop and bulk-indexes them into `mta_subway_realtime`
4. Offers ideas for keeping the data fresh (see the end of the notebook)

> ⚠️ This is **live, ephemeral** data - each pull is a snapshot. Re-run to refresh. Join it with `mta_subway_stops` from the schedule notebook to turn stop IDs into station names.

---
> **Data source:** MTA GTFS-Realtime subway feeds (public, no key).

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests gtfs-realtime-bindings protobuf --quiet

In [ ]:
import requests
from datetime import datetime, timezone
from google.transit import gtfs_realtime_pb2
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. The feeds
Each URL carries a group of lines. No API key needed - just GET the protobuf.

In [ ]:
BASE = 'https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2F'
FEEDS = {
    '123456S':  BASE + 'gtfs',
    'ACE':      BASE + 'gtfs-ace',
    'BDFM':     BASE + 'gtfs-bdfm',
    'G':        BASE + 'gtfs-g',
    'JZ':       BASE + 'gtfs-jz',
    'NQRW':     BASE + 'gtfs-nqrw',
    'L':        BASE + 'gtfs-l',
    'SIR':      BASE + 'gtfs-si',
}
print('Feeds configured:', ', '.join(FEEDS))

## 3. Fetch and parse the feeds into arrival documents

In [ ]:
def fetch_feed(name, url):
    feed = gtfs_realtime_pb2.FeedMessage()
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    feed.ParseFromString(r.content)

    docs = []
    for entity in feed.entity:
        if not entity.HasField('trip_update'):
            continue
        tu = entity.trip_update
        route = tu.trip.route_id
        trip_id = tu.trip.trip_id
        for stu in tu.stop_time_update:
            arr = stu.arrival.time if stu.HasField('arrival') else None
            if not arr:
                continue
            docs.append({
                'feed_group':   name,
                'route_id':     route,
                'trip_id':      trip_id,
                'stop_id':      stu.stop_id,
                'arrival_time': datetime.fromtimestamp(arr, tz=timezone.utc).isoformat(),
                'fetched_at':   datetime.now(tz=timezone.utc).isoformat(),
            })
    return docs

all_docs = []
for name, url in FEEDS.items():
    try:
        d = fetch_feed(name, url)
        all_docs.extend(d)
        print(f'  {name:8s} {len(d)} upcoming arrivals')
    except Exception as e:
        print(f'  {name:8s} ⚠️  {e}')

print(f'\n✅ Parsed {len(all_docs)} arrival predictions')

## 4. Create index and ingest the snapshot

In [ ]:
INDEX = 'mta_subway_realtime'
if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
es.indices.create(index=INDEX, body={'mappings': {'properties': {
    'feed_group':   {'type': 'keyword'},
    'route_id':     {'type': 'keyword'},
    'trip_id':      {'type': 'keyword'},
    'stop_id':      {'type': 'keyword'},
    'arrival_time': {'type': 'date'},
    'fetched_at':   {'type': 'date'},
}}})

helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in all_docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {len(all_docs)} arrivals into {INDEX}')

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, size=0, body={
    'aggs': {'by_route': {'terms': {'field': 'route_id', 'size': 30}}}
})
print(f"Total arrival predictions: {resp['hits']['total']['value']}")
print('\nUpcoming arrivals by route:')
for b in resp['aggregations']['by_route']['buckets']:
    print(f"  {b['key']:4s} {b['doc_count']}")

# Next few L trains (stop IDs; join with mta_subway_stops to get names)
resp = es.search(index=INDEX, size=5, body={
    'query': {'term': {'route_id': 'L'}},
    'sort': [{'arrival_time': 'asc'}]
})
print('\nNext L-train arrivals:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s['arrival_time']}  stop {s['stop_id']}")

## 6. Ideas: handling live data (try these yourself!)

This notebook loads a one-time snapshot. Here are some more idiomatic ways to handle a live feed in Elastic:

1. **Deterministic `_id` with upserts** (small change, same snapshot behavior)
   Use `_id = f"{trip_id}:{stop_id}"` in the bulk action. Each poll overwrites the same documents in place, so there's no delete step and no gap. Stale trips age out via a `fetched_at` range filter at query time, or a periodic `delete_by_query` where `fetched_at < now-2m`.

2. **Time-series data stream** (best fit if you want history)
   Index predictions as append-only events into a data stream, with `@timestamp` set to the fetch time. Query the latest state with a time-range filter, or with ES|QL using `INLINE STATS` or a `LAST`-style aggregation per trip and stop. You get arrival-prediction drift and delay analysis for free, and ILM or a retention setting expires old data automatically. For a hack-night demo, "how did the predicted arrival for this train change over the last 10 minutes" is a good story.

3. **Replace the notebook loop with a real ingest pipeline**
   Run the poller as a small script, an Elastic Agent custom integration, or a scheduled job, instead of a loop in a notebook cell. That's a better fit for live data, though more setup than a hack night warrants.

4. **Use the vehicle positions**
   This notebook only reads `trip_update` entities. The same feed has `vehicle` entities with current position and stop status. Indexing those with a `geo_point` makes a live map possible in Kibana.

---
## ✅ Index loaded
`mta_subway_realtime` holds the latest snapshot. Join `stop_id` with `mta_subway_stops` (schedule notebook) for station names to build a live-status view of the system.